In [1]:
from alpha_vantage.timeseries import TimeSeries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import io
import os
import time
import numpy as np

In [2]:
with open("api_key.txt") as file:
    API_key = file.read()

API_key = API_key.strip()
API_key

'N0YVA5RVTIR2D1OL'

In [3]:
pd.options.display.max_columns = None

## Using Alpha vantage python library

In [ ]:
#ts1 = TimeSeries(key=API_key)

In [ ]:
#ts1.get_monthly('AAPL')

({'2026-04-23': {'1. open': '254.0800',
   '2. high': '275.7700',
   '3. low': '245.7000',
   '4. close': '273.4300',
   '5. volume': '665999564'},
  '2026-03-31': {'1. open': '262.4100',
   '2. high': '266.5300',
   '3. low': '245.5100',
   '4. close': '253.7900',
   '5. volume': '900035757'},
  '2026-02-27': {'1. open': '260.0300',
   '2. high': '280.9050',
   '3. low': '255.4500',
   '4. close': '264.1800',
   '5. volume': '988325816'},
  '2026-01-30': {'1. open': '272.2550',
   '2. high': '277.8400',
   '3. low': '243.4200',
   '4. close': '259.4800',
   '5. volume': '1036170325'},
  '2025-12-31': {'1. open': '278.0100',
   '2. high': '288.6200',
   '3. low': '266.9500',
   '4. close': '271.8600',
   '5. volume': '922283649'},
  '2025-11-28': {'1. open': '270.4200',
   '2. high': '280.3800',
   '3. low': '265.3200',
   '4. close': '278.8500',
   '5. volume': '876481453'},
  '2025-10-31': {'1. open': '255.0400',
   '2. high': '277.3200',
   '3. low': '244.0000',
   '4. close': '270.

## Using Alpha vantage api

In [23]:
def return_json(url):
    # replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
    url = url
    r = requests.get(url)
    data = r.json()
    return data

In [24]:
def query_all_statements(ticker,query_func = None):
    """
    """
    if query_func != None:
        function_names= query_func
    else:
        function_names= {   "INCOME_STATEMENT":"annualReports",
                            "BALANCE_SHEET":"annualReports",
                            "CASH_FLOW":"annualReports",
                            "OVERVIEW":"EMPTY",
                            "DIVIDENDS":"data",
                            "SPLITS":"data",
                            "SHARES_OUTSTANDING":"data",
                            "EARNINGS":"annualEarnings",
                            "EARNINGS_ESTIMATES":"estimates"
                            }

    for func in function_names:
        url = f'https://www.alphavantage.co/query?function={func}&symbol={ticker}&apikey={API_key}'
        return_statement = return_json(url)

        # 1. Check for API limit or Error messages
        if "Information" in return_statement:
            print(f"⚠️ API Limit hit on {func}. Skipping...")
            time.sleep(60) # Wait a full minute if limited
            continue

        if func in ["INCOME_STATEMENT","BALANCE_SHEET","CASH_FLOW"]:
            print(f"key is {func}, value is {function_names[func]}")
            df = pd.json_normalize(return_statement.get(function_names[func])).set_index("fiscalDateEnding")
        elif func == "OVERVIEW":
            print(f"key is {func}, value is {function_names[func]}")
            df = pd.json_normalize(return_statement).set_index("Symbol")
        else:
            print(f"key is {func}, value is {function_names[func]}")
            df = pd.json_normalize(return_statement.get(function_names[func]))

        path = f"stocks/{ticker}"

        if not(os.path.exists(path)):
            os.mkdir(path)
        df.to_csv(f"./{path}/{func}.csv")
        time.sleep(15)

In [34]:
query_all_statements("NVDA")

key is INCOME_STATEMENT, value is annualReports
key is BALANCE_SHEET, value is annualReports
key is CASH_FLOW, value is annualReports
key is OVERVIEW, value is EMPTY
key is DIVIDENDS, value is data
key is SPLITS, value is data
key is SHARES_OUTSTANDING, value is data
⚠️ API Limit hit on EARNINGS. Skipping...
⚠️ API Limit hit on EARNINGS_ESTIMATES. Skipping...


KeyboardInterrupt: 

In [40]:
url = 'https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers=AAPL&apikey={API_key}'
appl_sentiment = return_json(url)

In [41]:
appl_sentiment

{'items': '50',
 'sentiment_score_definition': 'x <= -0.35: Bearish; -0.35 < x <= -0.15: Somewhat-Bearish; -0.15 < x < 0.15: Neutral; 0.15 <= x < 0.35: Somewhat_Bullish; x >= 0.35: Bullish',
 'relevance_score_definition': '0 < x <= 1, with a higher score indicating higher relevance.',
 'feed': [{'title': 'Why Is TD Cowen Doubling Down on Amazon (AMZN), Microsoft (MSFT), and Apple (AAPL) ahead of Earnings Next Week',
   'url': 'https://www.tipranks.com/news/why-is-td-cowen-doubling-down-on-amazon-amzn-microsoft-msft-and-apple-aapl-ahead-of-earnings-next-week',
   'time_published': '20260424T153952',
   'authors': ['Radhika Saraogi'],
   'summary': "TD Cowen is bullish on Amazon, Apple, and Microsoft ahead of their upcoming earnings reports, citing accelerated AI adoption, robust cloud demand, and strong product cycles as key drivers for durable growth. The firm believes the market is underestimating the AI momentum of these Big Tech giants and foresees significant revenue streams from G

In [4]:
df1 = pd.read_csv('./stocks/APPL/OVERVIEW.csv')
df1.head()

,Symbol,AssetType,Name,Description,CIK,Exchange,Currency,Country,Sector,Industry,Address,OfficialSite,FiscalYearEnd,LatestQuarter,MarketCapitalization,EBITDA,PERatio,PEGRatio,BookValue,DividendPerShare,DividendYield,EPS,RevenuePerShareTTM,ProfitMargin,OperatingMarginTTM,ReturnOnAssetsTTM,ReturnOnEquityTTM,RevenueTTM,GrossProfitTTM,DilutedEPSTTM,QuarterlyEarningsGrowthYOY,QuarterlyRevenueGrowthYOY,AnalystTargetPrice,AnalystRatingStrongBuy,AnalystRatingBuy,AnalystRatingHold,AnalystRatingSell,AnalystRatingStrongSell,TrailingPE,ForwardPE,PriceToSalesRatioTTM,PriceToBookRatio,EVToRevenue,EVToEBITDA,Beta,52WeekHigh,52WeekLow,50DayMovingAverage,200DayMovingAverage,SharesOutstanding,SharesFloat,PercentInsiders,PercentInstitutions,DividendDate,ExDividendDate
0,AAPL,Common Stock,Apple Inc,Apple Inc. is an American multinational techno...,320193,NASDAQ,USD,USA,TECHNOLOGY,CONSUMER ELECTRONICS,"ONE APPLE PARK WAY, CUPERTINO, CA, UNITED STAT...",https://www.apple.com,September,2025-12-31,3979469914000,152901992000,34.35,2.44,6.0,1.03,0.0038,7.89,29.3,0.27,0.354,0.244,1.52,435617006000,206157005000,7.89,0.183,0.157,297.71,6,25,15,1,1,34.35,31.75,9.14,45.12,9.19,26.18,1.109,288.35,192.41,260.15,253.64,14681140000,14656035000,1.64,65.225,2026-02-12,2026-02-09


In [6]:
df2 = pd.read_csv('./stocks/MSFT/OVERVIEW.csv')
df2.head()

,Symbol,AssetType,Name,Description,CIK,Exchange,Currency,Country,Sector,Industry,Address,OfficialSite,FiscalYearEnd,LatestQuarter,MarketCapitalization,EBITDA,PERatio,PEGRatio,BookValue,DividendPerShare,DividendYield,EPS,RevenuePerShareTTM,ProfitMargin,OperatingMarginTTM,ReturnOnAssetsTTM,ReturnOnEquityTTM,RevenueTTM,GrossProfitTTM,DilutedEPSTTM,QuarterlyEarningsGrowthYOY,QuarterlyRevenueGrowthYOY,AnalystTargetPrice,AnalystRatingStrongBuy,AnalystRatingBuy,AnalystRatingHold,AnalystRatingSell,AnalystRatingStrongSell,TrailingPE,ForwardPE,PriceToSalesRatioTTM,PriceToBookRatio,EVToRevenue,EVToEBITDA,Beta,52WeekHigh,52WeekLow,50DayMovingAverage,200DayMovingAverage,SharesOutstanding,SharesFloat,PercentInsiders,PercentInstitutions,DividendDate,ExDividendDate
0,MSFT,Common Stock,Microsoft Corporation,Microsoft Corporation is an American multinati...,789019,NASDAQ,USD,USA,TECHNOLOGY,SOFTWARE - INFRASTRUCTURE,"ONE MICROSOFT WAY, REDMOND, WA, UNITED STATES,...",https://www.microsoft.com,June,2025-12-31,3157422768000,175258993000,26.6,1.339,52.62,3.48,0.0082,15.97,41.1,0.39,0.471,0.149,0.344,305453007000,209498997000,15.97,0.598,0.167,572.67,10,45,3,0,0,26.6,22.12,10.34,8.07,10.22,16.58,1.107,552.24,356.28,394.52,469.71,7425629000,7414788000,0.079,75.883,2026-06-11,2026-05-21


In [7]:
df3 = pd.read_csv('./stocks/TSLA/OVERVIEW.csv')
df3.head()

,Symbol,AssetType,Name,Description,CIK,Exchange,Currency,Country,Sector,Industry,Address,OfficialSite,FiscalYearEnd,LatestQuarter,MarketCapitalization,EBITDA,PERatio,PEGRatio,BookValue,DividendPerShare,DividendYield,EPS,RevenuePerShareTTM,ProfitMargin,OperatingMarginTTM,ReturnOnAssetsTTM,ReturnOnEquityTTM,RevenueTTM,GrossProfitTTM,DilutedEPSTTM,QuarterlyEarningsGrowthYOY,QuarterlyRevenueGrowthYOY,AnalystTargetPrice,AnalystRatingStrongBuy,AnalystRatingBuy,AnalystRatingHold,AnalystRatingSell,AnalystRatingStrongSell,TrailingPE,ForwardPE,PriceToSalesRatioTTM,PriceToBookRatio,EVToRevenue,EVToEBITDA,Beta,52WeekHigh,52WeekLow,50DayMovingAverage,200DayMovingAverage,SharesOutstanding,SharesFloat,PercentInsiders,PercentInstitutions,DividendDate,ExDividendDate
0,TSLA,Common Stock,Tesla Inc,"Tesla, Inc. is an American electric vehicle an...",1318605,NASDAQ,USD,USA,CONSUMER CYCLICAL,AUTO MANUFACTURERS,"1 TESLA ROAD, AUSTIN, TX, UNITED STATES, 78725",https://www.tesla.com,December,2026-03-31,1412227269000,11094000000,344.97,5.19,21.9,NaN,NaN,1.09,30.31,0.0395,0.042,0.0223,0.049,97878999000,18660999000,1.09,0.083,0.158,416.45,5,18,17,6,2,344.97,181.82,14.43,16.91,14.24,115.47,1.915,498.83,271,385.48,401.51,3755724000,2815929000,11.121,44.641,NaN,NaN


In [12]:
df = pd.concat([df1,df2,df3],ignore_index=True)
df.head()

,Symbol,AssetType,Name,Description,CIK,Exchange,Currency,Country,Sector,Industry,Address,OfficialSite,FiscalYearEnd,LatestQuarter,MarketCapitalization,EBITDA,PERatio,PEGRatio,BookValue,DividendPerShare,DividendYield,EPS,RevenuePerShareTTM,ProfitMargin,OperatingMarginTTM,ReturnOnAssetsTTM,ReturnOnEquityTTM,RevenueTTM,GrossProfitTTM,DilutedEPSTTM,QuarterlyEarningsGrowthYOY,QuarterlyRevenueGrowthYOY,AnalystTargetPrice,AnalystRatingStrongBuy,AnalystRatingBuy,AnalystRatingHold,AnalystRatingSell,AnalystRatingStrongSell,TrailingPE,ForwardPE,PriceToSalesRatioTTM,PriceToBookRatio,EVToRevenue,EVToEBITDA,Beta,52WeekHigh,52WeekLow,50DayMovingAverage,200DayMovingAverage,SharesOutstanding,SharesFloat,PercentInsiders,PercentInstitutions,DividendDate,ExDividendDate
0,AAPL,Common Stock,Apple Inc,Apple Inc. is an American multinational techno...,320193,NASDAQ,USD,USA,TECHNOLOGY,CONSUMER ELECTRONICS,"ONE APPLE PARK WAY, CUPERTINO, CA, UNITED STAT...",https://www.apple.com,September,2025-12-31,3979469914000,152901992000,34.35,2.440,6.00,1.03,0.0038,7.89,29.30,0.2700,0.354,0.2440,1.520,435617006000,206157005000,7.89,0.183,0.157,297.71,6,25,15,1,1,34.35,31.75,9.14,45.12,9.19,26.18,1.109,288.35,192.41,260.15,253.64,14681140000,14656035000,1.640,65.225,2026-02-12,2026-02-09
1,MSFT,Common Stock,Microsoft Corporation,Microsoft Corporation is an American multinati...,789019,NASDAQ,USD,USA,TECHNOLOGY,SOFTWARE - INFRASTRUCTURE,"ONE MICROSOFT WAY, REDMOND, WA, UNITED STATES,...",https://www.microsoft.com,June,2025-12-31,3157422768000,175258993000,26.60,1.339,52.62,3.48,0.0082,15.97,41.10,0.3900,0.471,0.1490,0.344,305453007000,209498997000,15.97,0.598,0.167,572.67,10,45,3,0,0,26.60,22.12,10.34,8.07,10.22,16.58,1.107,552.24,356.28,394.52,469.71,7425629000,7414788000,0.079,75.883,2026-06-11,2026-05-21
2,TSLA,Common Stock,Tesla Inc,"Tesla, Inc. is an American electric vehicle an...",1318605,NASDAQ,USD,USA,CONSUMER CYCLICAL,AUTO MANUFACTURERS,"1 TESLA ROAD, AUSTIN, TX, UNITED STATES, 78725",https://www.tesla.com,December,2026-03-31,1412227269000,11094000000,344.97,5.190,21.90,NaN,NaN,1.09,30.31,0.0395,0.042,0.0223,0.049,97878999000,18660999000,1.09,0.083,0.158,416.45,5,18,17,6,2,344.97,181.82,14.43,16.91,14.24,115.47,1.915,498.83,271.00,385.48,401.51,3755724000,2815929000,11.121,44.641,NaN,NaN


In [13]:
df.rename(columns = {"Currency":"currency_id",
                     "Symbol":"stock_id"},inplace=True)
df.head()


,stock_id,AssetType,Name,Description,CIK,Exchange,currency_id,Country,Sector,Industry,Address,OfficialSite,FiscalYearEnd,LatestQuarter,MarketCapitalization,EBITDA,PERatio,PEGRatio,BookValue,DividendPerShare,DividendYield,EPS,RevenuePerShareTTM,ProfitMargin,OperatingMarginTTM,ReturnOnAssetsTTM,ReturnOnEquityTTM,RevenueTTM,GrossProfitTTM,DilutedEPSTTM,QuarterlyEarningsGrowthYOY,QuarterlyRevenueGrowthYOY,AnalystTargetPrice,AnalystRatingStrongBuy,AnalystRatingBuy,AnalystRatingHold,AnalystRatingSell,AnalystRatingStrongSell,TrailingPE,ForwardPE,PriceToSalesRatioTTM,PriceToBookRatio,EVToRevenue,EVToEBITDA,Beta,52WeekHigh,52WeekLow,50DayMovingAverage,200DayMovingAverage,SharesOutstanding,SharesFloat,PercentInsiders,PercentInstitutions,DividendDate,ExDividendDate
0,AAPL,Common Stock,Apple Inc,Apple Inc. is an American multinational techno...,320193,NASDAQ,USD,USA,TECHNOLOGY,CONSUMER ELECTRONICS,"ONE APPLE PARK WAY, CUPERTINO, CA, UNITED STAT...",https://www.apple.com,September,2025-12-31,3979469914000,152901992000,34.35,2.440,6.00,1.03,0.0038,7.89,29.30,0.2700,0.354,0.2440,1.520,435617006000,206157005000,7.89,0.183,0.157,297.71,6,25,15,1,1,34.35,31.75,9.14,45.12,9.19,26.18,1.109,288.35,192.41,260.15,253.64,14681140000,14656035000,1.640,65.225,2026-02-12,2026-02-09
1,MSFT,Common Stock,Microsoft Corporation,Microsoft Corporation is an American multinati...,789019,NASDAQ,USD,USA,TECHNOLOGY,SOFTWARE - INFRASTRUCTURE,"ONE MICROSOFT WAY, REDMOND, WA, UNITED STATES,...",https://www.microsoft.com,June,2025-12-31,3157422768000,175258993000,26.60,1.339,52.62,3.48,0.0082,15.97,41.10,0.3900,0.471,0.1490,0.344,305453007000,209498997000,15.97,0.598,0.167,572.67,10,45,3,0,0,26.60,22.12,10.34,8.07,10.22,16.58,1.107,552.24,356.28,394.52,469.71,7425629000,7414788000,0.079,75.883,2026-06-11,2026-05-21
2,TSLA,Common Stock,Tesla Inc,"Tesla, Inc. is an American electric vehicle an...",1318605,NASDAQ,USD,USA,CONSUMER CYCLICAL,AUTO MANUFACTURERS,"1 TESLA ROAD, AUSTIN, TX, UNITED STATES, 78725",https://www.tesla.com,December,2026-03-31,1412227269000,11094000000,344.97,5.190,21.90,NaN,NaN,1.09,30.31,0.0395,0.042,0.0223,0.049,97878999000,18660999000,1.09,0.083,0.158,416.45,5,18,17,6,2,344.97,181.82,14.43,16.91,14.24,115.47,1.915,498.83,271.00,385.48,401.51,3755724000,2815929000,11.121,44.641,NaN,NaN


In [10]:
df["stock_id"] = np.arange(1,df.shape[0]+1)

In [17]:
df = df.replace({'currency_id':{"USD":1},
            'stock_id':{'AAPL':1,
                        'MSFT':2,
                        'TSLA':3}
            })
df.head()

,stock_id,AssetType,Name,Description,CIK,Exchange,currency_id,Country,Sector,Industry,Address,OfficialSite,FiscalYearEnd,LatestQuarter,MarketCapitalization,EBITDA,PERatio,PEGRatio,BookValue,DividendPerShare,DividendYield,EPS,RevenuePerShareTTM,ProfitMargin,OperatingMarginTTM,ReturnOnAssetsTTM,ReturnOnEquityTTM,RevenueTTM,GrossProfitTTM,DilutedEPSTTM,QuarterlyEarningsGrowthYOY,QuarterlyRevenueGrowthYOY,AnalystTargetPrice,AnalystRatingStrongBuy,AnalystRatingBuy,AnalystRatingHold,AnalystRatingSell,AnalystRatingStrongSell,TrailingPE,ForwardPE,PriceToSalesRatioTTM,PriceToBookRatio,EVToRevenue,EVToEBITDA,Beta,52WeekHigh,52WeekLow,50DayMovingAverage,200DayMovingAverage,SharesOutstanding,SharesFloat,PercentInsiders,PercentInstitutions,DividendDate,ExDividendDate
0,1,Common Stock,Apple Inc,Apple Inc. is an American multinational techno...,320193,NASDAQ,1,USA,TECHNOLOGY,CONSUMER ELECTRONICS,"ONE APPLE PARK WAY, CUPERTINO, CA, UNITED STAT...",https://www.apple.com,September,2025-12-31,3979469914000,152901992000,34.35,2.440,6.00,1.03,0.0038,7.89,29.30,0.2700,0.354,0.2440,1.520,435617006000,206157005000,7.89,0.183,0.157,297.71,6,25,15,1,1,34.35,31.75,9.14,45.12,9.19,26.18,1.109,288.35,192.41,260.15,253.64,14681140000,14656035000,1.640,65.225,2026-02-12,2026-02-09
1,2,Common Stock,Microsoft Corporation,Microsoft Corporation is an American multinati...,789019,NASDAQ,1,USA,TECHNOLOGY,SOFTWARE - INFRASTRUCTURE,"ONE MICROSOFT WAY, REDMOND, WA, UNITED STATES,...",https://www.microsoft.com,June,2025-12-31,3157422768000,175258993000,26.60,1.339,52.62,3.48,0.0082,15.97,41.10,0.3900,0.471,0.1490,0.344,305453007000,209498997000,15.97,0.598,0.167,572.67,10,45,3,0,0,26.60,22.12,10.34,8.07,10.22,16.58,1.107,552.24,356.28,394.52,469.71,7425629000,7414788000,0.079,75.883,2026-06-11,2026-05-21
2,3,Common Stock,Tesla Inc,"Tesla, Inc. is an American electric vehicle an...",1318605,NASDAQ,1,USA,CONSUMER CYCLICAL,AUTO MANUFACTURERS,"1 TESLA ROAD, AUSTIN, TX, UNITED STATES, 78725",https://www.tesla.com,December,2026-03-31,1412227269000,11094000000,344.97,5.190,21.90,NaN,NaN,1.09,30.31,0.0395,0.042,0.0223,0.049,97878999000,18660999000,1.09,0.083,0.158,416.45,5,18,17,6,2,344.97,181.82,14.43,16.91,14.24,115.47,1.915,498.83,271.00,385.48,401.51,3755724000,2815929000,11.121,44.641,NaN,NaN


In [32]:
df.rename(columns={
    "stock_id": "stock_id",
    "AssetType": "asset_type",
    "Name": "name",
    "Description": "description",
    "CIK": "cik",
    "Exchange": "exchange",
    "currency_id": "currency_id",
    "Country": "country",
    "Sector": "sector",
    "Industry": "industry",
    "Address": "address",
    "OfficialSite": "official_site",
    "FiscalYearEnd": "fiscal_year_end",
    "LatestQuarter": "latest_quarter",
    "MarketCapitalization": "market_capitalization",
    "EBITDA": "ebitda",
    "PERatio": "pe_ratio",
    "PEGRatio": "peg_ratio",
    "BookValue": "book_value",
    "DividendPerShare": "dividend_per_share",
    "DividendYield": "dividend_yield",
    "EPS": "eps",
    "RevenuePerShareTTM": "revenue_per_share_ttm",
    "ProfitMargin": "profit_margin",
    "OperatingMarginTTM": "operating_margin_ttm",
    "ReturnOnAssetsTTM": "return_on_assets_ttm",
    "ReturnOnEquityTTM": "return_on_equity_ttm",
    "RevenueTTM": "revenue_ttm",
    "GrossProfitTTM": "gross_profit_ttm",
    "DilutedEPSTTM": "diluted_eps_ttm",
    "QuarterlyEarningsGrowthYOY": "quarterly_earnings_growth_yoy",
    "QuarterlyRevenueGrowthYOY": "quarterly_revenue_growth_yoy",
    "AnalystTargetPrice": "analyst_target_price",
    "AnalystRatingStrongBuy": "analyst_rating_strong_buy",
    "AnalystRatingBuy": "analyst_rating_buy",
    "AnalystRatingHold": "analyst_rating_hold",
    "AnalystRatingSell": "analyst_rating_sell",
    "AnalystRatingStrongSell": "analyst_rating_strong_sell",
    "TrailingPE": "trailing_pe",
    "ForwardPE": "forward_pe",
    "PriceToSalesRatioTTM": "price_to_sales_ratio_ttm",
    "PriceToBookRatio": "price_to_book_ratio",
    "EVToRevenue": "ev_to_revenue",
    "EVToEBITDA": "ev_to_ebitda",
    "Beta": "beta",
    "52WeekHigh": "high_52_week",
    "52WeekLow": "low_52_week",
    "50DayMovingAverage": "moving_average_50_day",
    "200DayMovingAverage": "moving_average_200_day",
    "SharesOutstanding": "shares_outstanding",
    "SharesFloat": "shares_float",
    "PercentInsiders": "percent_insiders",
    "PercentInstitutions": "percent_institutions",
    "DividendDate": "dividend_date",
    "ExDividendDate": "ex_dividend_date"
},inplace=True)

In [33]:
df.to_csv('../stock_db/data/overview.csv',index=False)